**03 · Comparación de particiones y evolución V0–V2**

Ejecutar desde el repositorio point-field con `python -m pip install -e ".[learning,bench,dev]"` y Jupyter. Las salidas nuevas se guardan en `runs/tesis_figuras/`. No reemplazar resultados históricos sin comprobar el protocolo.

In [ ]:
from pathlib import Path
import sys, subprocess, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'benchmarks'/'common.py').exists() and (p/'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz de point-field o una subcarpeta del repositorio.")
sys.path.insert(0, str(ROOT/'benchmarks'))
OUT = ROOT/'runs'/'tesis_figuras'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'figure.dpi': 120, 'savefig.dpi': 300, 'pdf.fonttype': 42})
COLORS = ['#21618c','#c65036','#63933e','#8b65a5','#c49a38','#458e91']
def save(fig, name):
    for ext in ('pdf','png'):
        fig.savefig(OUT/f'{name}.{ext}', bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
def points(ax, X, y=None):
    if y is None: ax.scatter(*X.T, s=5, c=COLORS[0], linewidths=0)
    else:
        for j,c in enumerate(np.unique(y)):
            ax.scatter(*X[y==c].T, s=5, c='#b9b9b9' if c==-1 else COLORS[j%len(COLORS)], linewidths=0)
    ax.set_xticks([]); ax.set_yticks([])
    if X.shape[1]==2: ax.set_aspect('equal', adjustable='box')
    else: ax.set_zticks([]); ax.set_box_aspect((1,1,1)); ax.view_init(elev=23, azim=35)
def run_script(name, *args):
    subprocess.run([sys.executable, str(ROOT/'benchmarks'/name), *map(str,args)], cwd=ROOT, check=True)
print('Repositorio:', ROOT, '\nSalidas:', OUT)
try:
    commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
except (OSError, subprocess.CalledProcessError):
    commit='No disponible'
print('Commit:',commit)

**1. Configuración del benchmark.** Se importa common.py para mantener hiperparámetros y las semillas del repositorio. Algunos métodos reciben el K verdadero; las etiquetas se usan en la evaluación.

In [ ]:
from common import MAIN_CASES,HELD_OUT_CASES,METHODS,VARIANTS,EVAL_SEEDS,load_case,timed_fit_predict
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score
methods=list(METHODS)
short=dict(zip(methods,['K-Means','DBSCAN','OPTICS','Espectral kNN','V0: global','V1: local','V2: fase']))
print({m:METHODS[m][3] for m in methods})

**2. Panel de comparación.** Una fila por escenario; columna de referencia y predicciones. ARI corresponde a una sola realización.

In [ ]:
def panel(cases,methods,variant,name):
    fig,axs=plt.subplots(len(cases),len(methods)+1,figsize=(2.0*(len(methods)+1),2.05*len(cases)),squeeze=False,constrained_layout=True)
    rows=[]
    for i,c in enumerate(cases):
        ds=load_case(c,variant,100);points(axs[i,0],ds.X,ds.y);axs[i,0].set_ylabel(c.replace('_',' '));axs[i,0].set_title('Referencia' if i==0 else '')
        for j,m in enumerate(methods,1):
            labels,dt=timed_fit_predict(METHODS[m][0](ds.n_groups),ds.X)
            ari=adjusted_rand_score(ds.y,labels);nmi=normalized_mutual_info_score(ds.y,labels)
            points(axs[i,j],ds.X,labels);axs[i,j].set_title((short[m]+'\n' if i==0 else '')+f'ARI {ari:.2f}')
            rows.append(dict(case=c,variant=variant,seed=100,method=m,ARI=ari,NMI=nmi,time_s=dt))
    pd.DataFrame(rows).to_csv(OUT/f'{name}.csv',index=False);save(fig,name)
panel(MAIN_CASES,methods[:5],'base','03_inicial')
panel(MAIN_CASES+HELD_OUT_CASES,methods[-3:],'base','03_evolucion')

**3. Benchmark completo opcional.** Al habilitarlo, se ejecutan las realizaciones de evaluación del protocolo público. No se modifica el CSV histórico; no incluye automáticamente UMAP ni el protocolo extendido del PDF.

In [ ]:
RUN_FULL=False
if RUN_FULL:
    rows=[]
    for c in MAIN_CASES+HELD_OUT_CASES:
        for v in VARIANTS:
            for seed in EVAL_SEEDS:
                ds=load_case(c,v,seed)
                for m in methods:
                    lab,dt=timed_fit_predict(METHODS[m][0](ds.n_groups),ds.X)
                    rows.append(dict(case=c,held_out=c in HELD_OUT_CASES,variant=v,seed=seed,method=m,ARI=adjusted_rand_score(ds.y,lab),NMI=normalized_mutual_info_score(ds.y,lab),time_s=dt))
    pd.DataFrame(rows).to_csv(OUT/'03_clustering_eval_nuevo.csv',index=False)
else: print('RUN_FULL=False: solo se generaron paneles de semilla 100.')